# Gaussian Processes

_Classical ML (beyond the cheat sheet)_

**A distribution over functions: predictions come with honest error bars.**

Most models give you one prediction. A Gaussian Process (GP) gives a prediction and its uncertainty.

     Instead of fitting one curve, a GP keeps a whole probability distribution over all curves that could fit.

---

This notebook is a practice scaffold for the **Gaussian Processes** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np, matplotlib.pyplot as plt

## Reference implementation — scikit-learn

In [ ]:
import numpy as np
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, WhiteKernel

rng = np.random.RandomState(0)
X = np.sort(rng.uniform(-3, 3, 12)).reshape(-1, 1)
y = np.sin(X).ravel() + 0.1 * rng.randn(12)

kernel = 1.0 * RBF(length_scale=1.0) + WhiteKernel(noise_level=0.01)
gp = GaussianProcessRegressor(kernel=kernel, random_state=0,
                              normalize_y=True).fit(X, y)

print("learned kernel:", gp.kernel_)
print("log marginal likelihood:", round(gp.log_marginal_likelihood_value_, 3))

Xtest = np.array([[0.0], [10.0]])   # one point near data, one far away
mean, std = gp.predict(Xtest, return_std=True)
print("at x=0  : mean=%.3f  std=%.3f" % (mean[0], std[0]))
print("at x=10 : mean=%.3f  std=%.3f" % (mean[1], std[1]))
print("-> std is far larger where there is no data")

## Visualize the data & results

_Fitting diabetes progression from BMI: how confident is the model where data is sparse?_

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_diabetes
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import (
    ConstantKernel, RBF, WhiteKernel)

dia = load_diabetes()
x = dia.data[:, 2]            # BMI feature (mean-centered, scaled)
y = dia.target.astype(float)

rng = np.random.RandomState(0)
idx = np.sort(rng.choice(len(x), 40, replace=False))
X = x[idx].reshape(-1, 1)
yo = y[idx]

kernel = (ConstantKernel(1.0) * RBF(length_scale=0.1)
          + WhiteKernel(noise_level=1.0))
gp = GaussianProcessRegressor(kernel=kernel, random_state=0,
                              normalize_y=True).fit(X, yo)

xs = np.linspace(x.min() - 0.02, x.max() + 0.02, 50).reshape(-1, 1)
mean, std = gp.predict(xs, return_std=True)

plt.scatter(X.ravel(), yo, c="#ffb454", s=30, zorder=3)
plt.plot(xs.ravel(), mean, c="#4ea1ff")
plt.fill_between(xs.ravel(), mean - 2 * std, mean + 2 * std,
                 color="#7ee787", alpha=0.3)
plt.title("GP regression on Diabetes (BMI vs progression)")
plt.xlabel("BMI (mean-centered, scaled)")
plt.ylabel("disease progression")
plt.show()